# Sensor Fusion — One EKF for **Real** Lidar + Radar (nuScenes)

We track a real vehicle by fusing **genuine lidar and radar returns** from the nuScenes dataset. One Extended Kalman Filter, one state `[px, py, vx, vy]`, corrected by whichever real sensor speaks next.

**A Kalman filter *is* late (measurement-level) fusion.** It fuses across **sensors** (lidar's precise position + radar's Doppler velocity) and across **time** (the motion model). The detection — *which returns belong to this car* — is given by the nuScenes annotation, so we focus entirely on the estimator.

| Sensor | Real measurement | Model |
|---|---|---|
| Lidar | centroid of real points in the box → `(x,y)` | linear |
| Radar | real return → `(range, bearing, Doppler)` about the moving ego | **non-linear** → Jacobian → *Extended* KF |


## 0. Setup (Colab)

The two real tracks are committed, so nothing to download to run the filter.


In [ ]:
# On Colab:
# !git clone https://github.com/Jeremy26/kalman_filters_course.git
# %cd kalman_filters_course/ekf_sensor_fusion
# !pip install -e '.[viz]' -q
import sys; sys.path.insert(0,'src')
import numpy as np, matplotlib.pyplot as plt
from kf_fusion import load_track, run_fusion


## 1. Load a real track

`track_437fe13d` is a real car: 5 lidar + 22 radar returns over 11 s. Lidar is *sparse* here — remember that.


In [ ]:
track = load_track('data/track_437fe13d.npz')
print(track.instance[:8], track.category, '—', len(track.measurements), 'real measurements')
lidar = [m for m in track.measurements if m.sensor=='lidar']
radar = [m for m in track.measurements if m.sensor=='radar']
print(f'{len(lidar)} lidar, {len(radar)} radar')
print('example radar z = [range, bearing, range-rate] =', np.round(radar[0].z,2))


## 2. What's real about the measurements?

Lidar gives position but no velocity. Radar's polar model is non-linear in the state, about the **moving** ego origin — that's the Jacobian, the 'E' in EKF.


In [ ]:
from kf_fusion import models
x = np.array([1310., 1040., 6., -3.]); sensor = radar[0].sensor_pos
print('h(x) =', np.round(models.radar_measurement(x, sensor),2), ' [range, bearing, range-rate]')
print('Jacobian Hj =\n', np.round(models.radar_jacobian(x, sensor),3))


## 3. Fuse — and see why it matters

Run the same real timeline three ways. A disabled sensor still advances time (predict-only), so lidar-only must *coast* through the long gaps between its 5 returns.


In [ ]:
fused      = run_fusion(track).summary()
lidar_only = run_fusion(track, use_radar=False).summary()
radar_only = run_fusion(track, use_lidar=False).summary()
for name,r in [('FUSED',fused),('LIDAR only',lidar_only),('RADAR only',radar_only)]:
    print(f"{name:11s} pos RMSE={r['rmse_pos']:5.1f} m   vel RMSE={r['rmse_vel']:.1f} m/s")
print('\nWith lidar this sparse, lidar-only is lost; radar carries the track and fusion is best.')


## 4. See the track

Estimate vs. real ground truth.


In [ ]:
res = run_fusion(track)
est, gt = res.estimates, res.ground_truth
plt.figure(figsize=(8,6))
plt.plot(gt[:,0], gt[:,1], 'g-', lw=2, label='ground truth')
plt.plot(est[:,0], est[:,1], 'b.--', label='fused EKF estimate')
lx=[m.z[0] for m in lidar]; ly=[m.z[1] for m in lidar]
plt.scatter(lx,ly,c='gold',ec='k',zorder=5,label='real lidar returns')
plt.axis('equal'); plt.legend(); plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title('Real vehicle track (nuScenes)'); plt.show()


## 5. Is the filter honest? (NIS consistency)

A good filter's innovations stay under the 95% chi-square bound ~95% of the time — it's neither over- nor under-confident.


In [ ]:
dense = load_track('data/track_ed634e83.npz')
s = run_fusion(dense).summary()
print(f"NIS under 95% bound — lidar {s['nis_lidar_below_95']:.2f}, radar {s['nis_radar_below_95']:.2f} (target ~0.95)")
print(f"pos RMSE {s['rmse_pos']:.2f} m, vel RMSE {s['rmse_vel']:.2f} m/s")


## 6. Export to Foxglove

Write an `.mcap` and open it at app.foxglove.dev with `layouts/ekf_fusion.json`. You'll see the **real lidar points**, the measurement, the estimate + breathing covariance, the **velocity arrow**, and a **1-second-ahead prediction**.


In [ ]:
run_fusion(dense, mcap_path='outputs/ekf_ed634e83.mcap')
print('wrote outputs/ekf_ed634e83.mcap')
# On Colab: from google.colab import files; files.download('outputs/ekf_ed634e83.mcap')


## 7. Your turn

1. **Tune the process noise** `run_fusion(track, noise_ax=, noise_ay=)` and watch NIS — find the value that keeps ~95% of innovations under the bound.
2. **Re-extract a different vehicle** with `kf_fusion.nuscenes_extract` (needs the dataset) and see how coverage changes the fusion story.
3. **Inspect the lidar bias**: plot `lidar return − ground-truth center`. Why is it biased toward one side? (Hint: which face does lidar see?)

> **Next course:** run *N* of these filters at once and decide which return belongs to which track — that's **data association**, the start of multi-object tracking.
